# CS 133: POS Tagging for Kankanaey Simple Sentences

This notebook tags each collected sentence with part-of-speech (POS) information using NLTK,
then sets up a place to transfer those tags onto the Kankanaey words for the context-free
grammar step.

**How to use this as a group:**
1. Everyone opens this notebook from the shared GitHub repo (`File > Open notebook > GitHub` tab in Colab).
2. The data (`List of Kankanaey Sentences.xlsx`) is read directly from the repo's raw GitHub URL, so
   nobody needs to re-upload the file to run this.
3. When you make changes, save back with `File > Save a copy in GitHub`, or use the git commands
   in the last cell if you cloned the repo into your Colab runtime.


## 1. Install and download NLTK resources

In [7]:
import nltk

# These downloads only need to run once per Colab session
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

## 2. Load the sentence data

Replace `RAW_URL` below with your own repo's raw file link once the XLSX is pushed to GitHub.
Format: `https://raw.githubusercontent.com/<username>/<repo>/<branch>/data/List of Kankanaey Sentences.xlsx`


In [11]:
import pandas as pd

RAW_URL = "https://raw.githubusercontent.com/WaterDispenzer/CS-133-Finals-Project/main/data/List_of_Kankanaey_Sentences.xlsx"

df = pd.read_excel(RAW_URL, engine="openpyxl")
print(df.shape)
df.head()


InvalidURL: URL can't contain control characters. '/WaterDispenzer/CS-133-Finals-Project/main/data/List of Kankanaey Sentences.xlsx' (found at least ' ')

## 3. POS-tag the English sentences

NLTK's tagger only supports English out of the box, so we tag the English translations first.
This gives us a reliable set of POS tags we can then line up against the Kankanaey words.

In [ ]:
from nltk import word_tokenize, pos_tag

def tag_english_sentence(sentence):
    tokens = word_tokenize(sentence)
    return pos_tag(tokens)

df['english_pos_tags'] = df['english'].apply(tag_english_sentence)

# Quick look at the first few
for i in range(3):
    print(df.loc[i, 'english'])
    print(df.loc[i, 'english_pos_tags'])
    print()


I love you.
[('I', 'PRP'), ('love', 'VBP'), ('you', 'PRP'), ('.', '.')]

The man went.
[('The', 'DT'), ('man', 'NN'), ('went', 'VBD'), ('.', '.')]

The woman will sing.
[('The', 'DT'), ('woman', 'NN'), ('will', 'MD'), ('sing', 'VB'), ('.', '.')]



## 4. Tokenize the Kankanaey sentences

We can split Kankanaey sentences into tokens the same way (simple whitespace/punctuation
tokenization). NLTK's tagger itself will not produce correct tags for Kankanaey words,
since it was trained only on English, so this step is only for splitting the sentence into
words to line up against the English tags next.

In [ ]:
df['kankanaey_tokens'] = df['kankanaey'].apply(word_tokenize)

for i in range(3):
    print(df.loc[i, 'kankanaey'])
    print(df.loc[i, 'kankanaey_tokens'])
    print()


Laylaydek sik-a
['Laylaydek', 'sik-a']

Inmey din lalaki
['Inmey', 'din', 'lalaki']

Mankanta din babai.
['Mankanta', 'din', 'babai', '.']



## 5. Align Kankanaey words to English POS tags

This is the part that needs a human in the loop. Kankanaey and English don't have the same
word order or word count per sentence, so a Kankanaey word can't just be matched to the English
word in the same position. For each sentence, a fluent speaker in the group should confirm which
Kankanaey word or affix corresponds to which POS-tagged English word.

The cell below builds an empty alignment table you can fill in, one row per sentence, with a
column for each Kankanaey token and a place to assign its POS tag. Export it to XLSX, divide the
rows among the group, and fill it in in Google Sheets or Excel, then re-import it for the next
step (grammar rule extraction).

In [ ]:
alignment_rows = []

for idx, row in df.iterrows():
    for token in row['kankanaey_tokens']:
        alignment_rows.append({
            'sentence_id': row['sentence_id'],
            'kankanaey_sentence': row['kankanaey'],
            'english_sentence': row['english'],
            'kankanaey_token': token,
            'assigned_pos_tag': ''  # to be filled in by a fluent speaker
        })

alignment_df = pd.DataFrame(alignment_rows)
alignment_df.to_excel('kankanaey_pos_alignment_template.xlsx', index=False)
alignment_df.head(10)


,sentence_id,kankanaey_sentence,english_sentence,kankanaey_token,assigned_pos_tag
0,1,Laylaydek sik-a,I love you.,Laylaydek,
1,1,Laylaydek sik-a,I love you.,sik-a,
2,2,Inmey din lalaki,The man went.,Inmey,
3,2,Inmey din lalaki,The man went.,din,
4,2,Inmey din lalaki,The man went.,lalaki,
5,3,Mankanta din babai.,The woman will sing.,Mankanta,
6,3,Mankanta din babai.,The woman will sing.,din,
7,3,Mankanta din babai.,The woman will sing.,babai,
8,3,Mankanta din babai.,The woman will sing.,.,
9,4,Wada di aso.,There is a dog.,Wada,


## 6. Save results back to the repo

Once tagging is done (either the English-only tags, or the filled-in Kankanaey alignment file),
save the outputs and push them back to GitHub so the whole group stays in sync.

If you cloned the repo into this Colab runtime with `git clone`, you can commit and push directly:


In [ ]:
# Only run this if you cloned the repo into this runtime (see setup instructions)
# df.to_excel('/content/YOUR-REPO/data/List of Kankanaey Sentences Tagged.xlsx', index=False)
# alignment_df.to_excel('/content/YOUR-REPO/data/kankanaey_pos_alignment_template.xlsx', index=False)

# !cd /content/YOUR-REPO && git add . && git commit -m "Add POS tagging output" && git push
